# Batch 06 — Exp02: `v4_self_reported_rating` (Self-Reported Perceived Difficulty 1–10)

Notebook mandiri untuk eksperimen pada paket **`v4_self_reported_rating`** (`window_15s_step_5s/good_acceptable.csv`, 1.693 window, 126 episode, 17 peserta).

**Rancangan Eksperimen (Bebas Kebocoran & Independen dari V3):**
- Pembagian deterministik berbasis cakupan metadata peserta: **Development (13 peserta)** untuk 5-fold `GroupKFold` seleksi model dan **Held-Out Test (4 peserta)** yang hanya dievaluasi **satu kali** oleh model pemenang terkunci.
- **Track A — Regresi Kontinu/Ordinal (`difficulty_rating` 1–10):** MAE, RMSE, $R^2$, Spearman $\rho$ (window & episode).
- **Track B — Klasifikasi 3-Kelas (`Low: 1–3`, `Med: 4–6`, `High: 7–10`):** Accuracy, Balanced Accuracy, Macro-F1 (window & episode).
- **Track C1 — Biner Ekstrem Lintas-Peserta (`Low: 1–3` vs `High: 7–10`):** ROC-AUC, Balanced Accuracy, Macro-F1.
- **Track C2 — Diagnostik Kalibrasi Dalam-Peserta (*Within-Participant Leave-One-Episode-Out*):** Menguji apakah sinyal fisiologis dapat dipelajari secara personal ketika seluruh window satu episode ditahan sekaligus (bebas kebocoran overlap).


In [1]:
# Cell 1: Konfigurasi Lingkungan, Seed, dan Verifikasi SHA-256 Dataset V4
import hashlib
import itertools
import json
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import platform
import random
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
if "ipykernel" not in os.sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    r2_score,
    recall_score,
    roc_auc_score,
    root_mean_squared_error,
)
from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
TIERS = ["Low", "Med", "High"]
TIER_TO_IDX = {t: i for i, t in enumerate(TIERS)}

def set_all_seeds(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_all_seeds(SEED)
torch.set_num_threads(2)

IN_COLAB = "google.colab" in os.sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = Path("/content/drive/MyDrive/Penelitian-Adaptive-IDE")
    DATASET_ROOT = PROJECT_ROOT / "models/batch-06/dataset"
else:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd.parent.parent,
        cwd.parent.parent.parent,
        Path(r"e:\Skripsi\ngodong\Adaptive-IDE"),
    ]
    PROJECT_ROOT = next((p for p in candidates if (p / "models/batch-06/dataset").exists()), cwd)
    DATASET_ROOT = PROJECT_ROOT / "models/batch-06/dataset"

V4_DIR = DATASET_ROOT / "v4_self_reported_rating"
OUTPUT_DIR = PROJECT_ROOT / "models/batch-06/outputs/exp02_v4_self_reported_rating"
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = [
    "window_15s_step_5s/good_acceptable.csv",
    "window_15s_step_5s/window_assignments.csv",
    "episode_ratings.csv",
]
file_hashes = {}
for rel in REQUIRED_FILES:
    p = V4_DIR / rel
    assert p.exists(), f"File wajib tidak ditemukan: {p}"
    file_hashes[rel] = hashlib.sha256(p.read_bytes()).hexdigest()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"V4_DIR       : {V4_DIR}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"DEVICE       : {DEVICE} | Python {platform.python_version()} | PyTorch {torch.__version__}")
print("SHA-256 dataset v4 terverifikasi (3/3 file).")


PROJECT_ROOT : E:\Skripsi\ngodong\Adaptive-IDE
V4_DIR       : E:\Skripsi\ngodong\Adaptive-IDE\models\batch-06\dataset\v4_self_reported_rating
OUTPUT_DIR   : E:\Skripsi\ngodong\Adaptive-IDE\models\batch-06\outputs\exp02_v4_self_reported_rating
DEVICE       : cpu | Python 3.10.11 | PyTorch 2.11.0+cpu
SHA-256 dataset v4 terverifikasi (3/3 file).


In [2]:
# Cell 2: Muat & Gabungkan CSV V4 + Pembagian Peserta Deterministik (Metadata-Only)
CORE10 = [
    "rmssd_ms",
    "sdnn_ms",
    "blink_rate_per_minute",
    "ear_left_mean",
    "ear_left_std",
    "ear_right_mean",
    "ear_right_std",
    "iris_size_estimate_mean",
    "iris_size_estimate_std",
    "iris_size_delta_mean",
]
CORE9 = [c for c in CORE10 if c != "iris_size_delta_mean"]
HEAD4 = ["head_pitch_mean", "head_pitch_std", "head_yaw_mean", "head_yaw_std"]
FEATURE_PROFILES = {
    "core10": CORE10,
    "core9": CORE9,
    "core9_head4": CORE9 + HEAD4,
    "core9_head4_base": CORE9 + HEAD4 + ["rmssd_baseline_ratio"],
}

raw_df = pd.read_csv(V4_DIR / "window_15s_step_5s/good_acceptable.csv")
wa_df = pd.read_csv(V4_DIR / "window_15s_step_5s/window_assignments.csv")
ep_df_raw = pd.read_csv(V4_DIR / "episode_ratings.csv")

assert raw_df["window_id"].is_unique, "window_id duplikat!"
df = raw_df.merge(
    wa_df[["window_id", "episode_id", "difficulty_rating"]].rename(columns={"difficulty_rating": "wa_rating"}),
    on="window_id",
    how="left",
    validate="1:1",
)
assert df["episode_id"].notna().all(), "Ada window tanpa episode_id!"
assert (df["difficulty_rating"] == df["wa_rating"]).all(), "Ketidakcocokan rating antara good_acceptable dan window_assignments!"
df = df.drop(columns=["wa_rating"])

assert np.isfinite(df[CORE10 + HEAD4].to_numpy(float)).all(), "Nilai non-finite pada CORE10/HEAD4!"

df["tier"] = pd.cut(df["difficulty_rating"], bins=[0, 3, 6, 10], labels=TIERS).astype(str)
df["window_start_dt"] = pd.to_datetime(df["window_start"])
df = df.sort_values(["session_id", "window_start_dt", "window_id"]).reset_index(drop=True)

# Pembagian peserta deterministik (13 Development, 4 Held-Out Test) murni dari cakupan metadata tier
ep_meta = df.groupby(["episode_id", "participant_id"], as_index=False).agg(
    rating=("difficulty_rating", "first"),
    tier=("tier", "first"),
    n_win=("window_id", "count"),
)
people = sorted(ep_meta["participant_id"].unique())
global_ep_prop = ep_meta["tier"].value_counts(normalize=True).reindex(TIERS, fill_value=0.0).to_numpy()

best_combo = None
best_score = (1e9, 1e9, ())
for combo in itertools.combinations(people, 4):
    te_set = set(combo)
    te_ep = ep_meta[ep_meta["participant_id"].isin(te_set)]
    dv_ep = ep_meta[~ep_meta["participant_id"].isin(te_set)]
    te_p_per_tier = [te_ep[te_ep["tier"] == t]["participant_id"].nunique() for t in TIERS]
    dv_p_per_tier = [dv_ep[dv_ep["tier"] == t]["participant_id"].nunique() for t in TIERS]
    # Syarat keras: minimal 3 peserta Test & 9 peserta Dev pada tiap tier, dan 20-28% episode di Test
    ep_frac = len(te_ep) / len(ep_meta)
    win_frac = te_ep["n_win"].sum() / ep_meta["n_win"].sum()
    if min(te_p_per_tier) < 3 or min(dv_p_per_tier) < 9:
        continue
    if not (0.20 <= ep_frac <= 0.28 and 0.18 <= win_frac <= 0.30):
        continue
    te_prop = te_ep["tier"].value_counts(normalize=True).reindex(TIERS, fill_value=0.0).to_numpy()
    l1_dist = float(np.abs(te_prop - global_ep_prop).sum())
    size_dev = abs(ep_frac - 0.24) + abs(win_frac - 0.24)
    key = (round(l1_dist, 6), round(size_dev, 6), combo)
    if key < best_score:
        best_score = key
        best_combo = combo

assert best_combo is not None, "Tidak ditemukan pembagian peserta V4 yang memenuhi syarat cakupan!"
test_people = set(best_combo)
df["split"] = np.where(df["participant_id"].isin(test_people), "Test", "Dev")

split_table = pd.DataFrame(
    [{"participant_id": p, "split": "Test" if p in test_people else "Dev"} for p in people]
)
split_table.to_csv(OUTPUT_DIR / "participant_splits_v4.csv", index=False)

assert df.groupby("participant_id")["split"].nunique().max() == 1
assert df.groupby("episode_id")["split"].nunique().max() == 1

print(f"Total rows: {len(df)} | Episodes: {df['episode_id'].nunique()} | Participants: {df['participant_id'].nunique()}")
for sp in ["Dev", "Test"]:
    sub = df[df["split"] == sp]
    t_str = ", ".join(
        f"{t}={len(sub[sub['tier']==t])}w/{sub[sub['tier']==t]['episode_id'].nunique()}ep/{sub[sub['tier']==t]['participant_id'].nunique()}p"
        for t in TIERS
    )
    print(f"  {sp:4s} ({sub['participant_id'].nunique():2d}p, {sub['episode_id'].nunique():3d}ep, {len(sub):4d}w): {t_str}")
print("Held-Out Test participants:", sorted(test_people))


Total rows: 1693 | Episodes: 126 | Participants: 17
  Dev  (13p,  99ep, 1328w): Low=391w/52ep/10p, Med=378w/29ep/9p, High=559w/18ep/9p
  Test ( 4p,  27ep,  365w): Low=121w/14ep/4p, Med=96w/8ep/4p, High=148w/5ep/3p
Held-Out Test participants: ['Athaya', 'Baithar', 'Sullivan', 'Tio']


In [3]:
# Cell 3: Preprocessor Bebas Kebocoran, Bobot Episode-Equalized, & Definisi Model
class V4SensorPreprocessor:
    """Imputasi median fitting-only (jika ada NaN seperti rmssd_baseline_ratio) + robust scaling."""
    def __init__(self, cols: list, mode: str = "global_robust"):
        self.cols = cols
        self.mode = mode
        self.g_med_ = None
        self.g_iqr_ = None

    def fit(self, fit_df: pd.DataFrame):
        vals = fit_df[self.cols].to_numpy(dtype=float)
        self.g_med_ = np.nanmedian(vals, axis=0)
        filled = np.where(np.isfinite(vals), vals, self.g_med_)
        q75, q25 = np.percentile(filled, [75, 25], axis=0)
        iqr = q75 - q25
        iqr[iqr <= 1e-9] = 1.0
        self.g_iqr_ = iqr
        return self

    def transform(self, target_df: pd.DataFrame) -> np.ndarray:
        vals = target_df[self.cols].to_numpy(dtype=float)
        vals = np.where(np.isfinite(vals), vals, self.g_med_)
        if self.mode == "global_robust":
            return np.clip((vals - self.g_med_) / self.g_iqr_, -10.0, 10.0)

        out = np.zeros_like(vals, dtype=float)
        if self.mode == "unsupervised_session_robust":
            for sid, idx in target_df.groupby("session_id").indices.items():
                block = vals[idx]
                if len(block) < 3:
                    out[idx] = (block - self.g_med_) / self.g_iqr_
                else:
                    med = np.median(block, axis=0)
                    q75, q25 = np.percentile(block, [75, 25], axis=0)
                    iqr = np.where((q75 - q25) > 1e-9, q75 - q25, self.g_iqr_)
                    out[idx] = (block - med) / iqr
            return np.clip(out, -10.0, 10.0)

        if self.mode == "causal_session_robust":
            for sid, idx in target_df.groupby("session_id").indices.items():
                idx_sorted = idx[np.argsort(target_df.iloc[idx]["window_start_dt"].to_numpy())]
                for pos, row_i in enumerate(idx_sorted):
                    if pos < 3:
                        out[row_i] = (vals[row_i] - self.g_med_) / self.g_iqr_
                    else:
                        hist = vals[idx_sorted[:pos]]
                        med = np.median(hist, axis=0)
                        q75, q25 = np.percentile(hist, [75, 25], axis=0)
                        iqr = np.where((q75 - q25) > 1e-9, q75 - q25, self.g_iqr_)
                        out[row_i] = (vals[row_i] - med) / iqr
            return np.clip(out, -10.0, 10.0)
        raise ValueError(self.mode)


def compute_v4_weights(sub_df: pd.DataFrame, target_col: str = None, mode: str = "episode_equalized_clipped") -> np.ndarray:
    """Bobot Train untuk menyeimbangkan kontribusi episode panjang (dan kelas bila klasifikasi)."""
    n = len(sub_df)
    if mode == "uniform":
        return np.ones(n, dtype=float)
    if mode == "class_balanced" and target_col is not None:
        vc = sub_df[target_col].value_counts()
        k = len(vc)
        w = sub_df[target_col].map(lambda c: n / (k * vc[c])).to_numpy(dtype=float)
        return w
    if mode == "episode_equalized_clipped":
        if target_col is None:
            # Regresi: setiap episode mendapat bobot setara, dipangkas maks 5x
            ep_sizes = sub_df.groupby("episode_id")["window_id"].transform("count").to_numpy(dtype=float)
            w = (n / sub_df["episode_id"].nunique()) / ep_sizes
            w = np.minimum(w, 5.0)
            return w * (n / w.sum())
        else:
            # Klasifikasi: seimbang per kelas lalu per episode, dipangkas maks 5x
            w = np.ones(n, dtype=float)
            classes = sub_df[target_col].unique()
            k = len(classes)
            for cls in classes:
                mask = (sub_df[target_col] == cls).to_numpy()
                g = sub_df.loc[mask]
                n_ep = g["episode_id"].nunique()
                ep_sz = g.groupby("episode_id")["window_id"].transform("count").to_numpy(dtype=float)
                wc = (n / (k * n_ep)) / ep_sz
                wc = np.minimum(wc, 5.0)
                wc = wc * ((n / k) / wc.sum())
                w[mask] = wc
            return w
    raise ValueError(mode)


def make_v4_episode_summary(window_df: pd.DataFrame, cols: list, prep: V4SensorPreprocessor) -> pd.DataFrame:
    X_norm = prep.transform(window_df)
    tmp = pd.DataFrame(X_norm, columns=cols, index=window_df.index)
    tmp["episode_id"] = window_df["episode_id"].to_numpy()
    tmp["participant_id"] = window_df["participant_id"].to_numpy()
    tmp["difficulty_rating"] = window_df["difficulty_rating"].to_numpy()
    tmp["tier"] = window_df["tier"].to_numpy()
    tmp["t_sec"] = (window_df["window_start_dt"] - window_df.groupby("episode_id")["window_start_dt"].transform("min")).dt.total_seconds()
    rows = []
    for eid, g in tmp.groupby("episode_id", sort=True):
        row = {
            "episode_id": eid,
            "participant_id": g["participant_id"].iloc[0],
            "difficulty_rating": float(g["difficulty_rating"].iloc[0]),
            "tier": g["tier"].iloc[0],
        }
        t = g["t_sec"].to_numpy(dtype=float)
        for c in cols:
            v = g[c].to_numpy(dtype=float)
            row[f"{c}_med"] = float(np.median(v))
            row[f"{c}_iqr"] = float(np.percentile(v, 75) - np.percentile(v, 25))
            slope = float(np.polyfit(t / 60.0, v, 1)[0]) if (len(v) >= 3 and np.std(t) > 1e-6) else 0.0
            row[f"{c}_slope"] = float(np.clip(slope, -10.0, 10.0))
        rows.append(row)
    return pd.DataFrame(rows)


class SmallMLP(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.LayerNorm(32),
            nn.SiLU(),
            nn.Dropout(0.30),
            nn.Linear(32, 16),
            nn.LayerNorm(16),
            nn.SiLU(),
            nn.Dropout(0.20),
            nn.Linear(16, out_dim),
        )

    def forward(self, x):
        return self.net(x)


def fit_predict_mlp(task: str, X_tr, y_tr, w_tr, groups_tr, X_eval, n_classes: int = 3, seeds=(42, 43, 44), max_epochs: int = 60):
    uniq_g = np.unique(groups_tr)
    rng_m = np.random.RandomState(SEED)
    perm_g = rng_m.permutation(uniq_g)
    n_mon = max(1, min(2, len(uniq_g) // 5))
    mon_mask = np.isin(groups_tr, perm_g[:n_mon])
    if mon_mask.sum() < 5 or (~mon_mask).sum() < 10:
        mon_mask = np.zeros(len(y_tr), dtype=bool)

    out_dim = 1 if task == "reg" else n_classes
    agg = np.zeros((len(X_eval), out_dim), dtype=float)

    for s in seeds:
        set_all_seeds(s)
        model = SmallMLP(X_tr.shape[1], out_dim).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-2)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)

        fit_idx = np.where(~mon_mask)[0]
        X_f = torch.tensor(X_tr[fit_idx], dtype=torch.float32, device=DEVICE)
        y_f = torch.tensor(y_tr[fit_idx], dtype=torch.float32 if task == "reg" else torch.long, device=DEVICE)
        w_f = torch.tensor(w_tr[fit_idx], dtype=torch.float32, device=DEVICE)
        n_fit = len(fit_idx)
        batch_size = 128

        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_loss = float("inf")
        patience, bad = 12, 0
        if mon_mask.any():
            X_m = torch.tensor(X_tr[mon_mask], dtype=torch.float32, device=DEVICE)
            y_m = torch.tensor(y_tr[mon_mask], dtype=torch.float32 if task == "reg" else torch.long, device=DEVICE)

        for _ in range(max_epochs):
            model.train()
            perm = torch.randperm(n_fit, device=DEVICE)
            for start in range(0, n_fit, batch_size):
                bidx = perm[start : start + batch_size]
                xb, yb, wb = X_f[bidx], y_f[bidx], w_f[bidx]
                opt.zero_grad()
                pred = model(xb)
                if task == "reg":
                    raw_l = nn.functional.smooth_l1_loss(pred.squeeze(-1), yb, reduction="none")
                else:
                    raw_l = nn.functional.cross_entropy(pred, yb, reduction="none")
                loss = (raw_l * wb).sum() / (wb.sum() + 1e-8)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                opt.step()
            sched.step()
            if mon_mask.any():
                model.eval()
                with torch.no_grad():
                    mp = model(X_m)
                    ml = float(
                        nn.functional.smooth_l1_loss(mp.squeeze(-1), y_m).item()
                        if task == "reg"
                        else nn.functional.cross_entropy(mp, y_m).item()
                    )
                if ml < best_loss - 1e-4:
                    best_loss = ml
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    bad = 0
                else:
                    bad += 1
                    if bad >= patience:
                        break
            else:
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            ev = model(torch.tensor(X_eval, dtype=torch.float32, device=DEVICE))
            if task == "reg":
                agg += np.clip(ev.cpu().numpy(), 1.0, 10.0)
            else:
                agg += torch.softmax(ev, dim=-1).cpu().numpy()
    agg /= len(seeds)
    return agg.squeeze(-1) if task == "reg" else agg


def fit_predict_reg(model_name, X_tr, y_tr, w_tr, groups_tr, X_eval):
    if model_name == "DummyMean":
        m = DummyRegressor(strategy="mean").fit(X_tr, y_tr)
        return np.clip(m.predict(X_eval), 1.0, 10.0)
    if model_name == "Ridge":
        m = Ridge(alpha=10.0, random_state=SEED).fit(X_tr, y_tr, sample_weight=w_tr)
        return np.clip(m.predict(X_eval), 1.0, 10.0)
    if model_name == "ExtraTreesReg":
        m = ExtraTreesRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.8, random_state=SEED, n_jobs=1)
        m.fit(X_tr, y_tr, sample_weight=w_tr)
        return np.clip(m.predict(X_eval), 1.0, 10.0)
    if model_name == "HistGBReg":
        m = HistGradientBoostingRegressor(
            max_iter=120, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=10,
            l2_regularization=1.0, early_stopping=False, random_state=SEED
        )
        m.fit(X_tr, y_tr, sample_weight=w_tr)
        return np.clip(m.predict(X_eval), 1.0, 10.0)
    if model_name == "PyTorchMLPReg":
        return fit_predict_mlp("reg", X_tr, y_tr, w_tr, groups_tr, X_eval)
    raise ValueError(model_name)


def fit_predict_clf(model_name, X_tr, y_tr, w_tr, groups_tr, X_eval, n_classes: int = 3):
    if model_name == "Dummy":
        m = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
        probs = m.predict_proba(X_eval)
    elif model_name == "LogisticRegression":
        m = LogisticRegression(C=0.5, max_iter=5000, random_state=SEED).fit(X_tr, y_tr, sample_weight=w_tr)
        probs = m.predict_proba(X_eval)
    elif model_name == "ExtraTrees":
        m = ExtraTreesClassifier(n_estimators=200, min_samples_leaf=5, max_features=0.8, random_state=SEED, n_jobs=1)
        m.fit(X_tr, y_tr, sample_weight=w_tr)
        probs = m.predict_proba(X_eval)
    elif model_name == "HistGradientBoosting":
        m = HistGradientBoostingClassifier(
            max_iter=120, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=10,
            l2_regularization=1.0, early_stopping=False, random_state=SEED
        )
        m.fit(X_tr, y_tr, sample_weight=w_tr)
        probs = m.predict_proba(X_eval)
    elif model_name == "PyTorchMLP":
        return fit_predict_mlp("clf", X_tr, y_tr, w_tr, groups_tr, X_eval, n_classes=n_classes)
    else:
        raise ValueError(model_name)

    full = np.zeros((len(X_eval), n_classes), dtype=float)
    for col_i, cls_v in enumerate(m.classes_):
        full[:, int(cls_v)] = probs[:, col_i]
    return full


def eval_regression(y_true, y_pred, episode_ids, participant_ids) -> dict:
    win_mae = float(mean_absolute_error(y_true, y_pred))
    win_rmse = float(root_mean_squared_error(y_true, y_pred))
    win_r2 = float(r2_score(y_true, y_pred))
    win_rho = float(stats.spearmanr(y_true, y_pred).statistic) if np.std(y_pred) > 1e-9 else 0.0

    ep_df = pd.DataFrame({"ep": episode_ids, "pid": participant_ids, "y": y_true, "p": y_pred}).groupby(["ep", "pid"], as_index=False).mean()
    ep_mae = float(mean_absolute_error(ep_df["y"], ep_df["p"]))
    ep_rmse = float(root_mean_squared_error(ep_df["y"], ep_df["p"]))
    ep_r2 = float(r2_score(ep_df["y"], ep_df["p"]))
    ep_rho = float(stats.spearmanr(ep_df["y"], ep_df["p"]).statistic) if np.std(ep_df["p"]) > 1e-9 else 0.0

    wp_rhos = []
    for _, g in ep_df.groupby("pid"):
        if len(g) >= 3 and g["y"].nunique() >= 2 and g["p"].nunique() >= 2:
            r = stats.spearmanr(g["y"], g["p"]).statistic
            if np.isfinite(r):
                wp_rhos.append(float(r))
    return {
        "n_windows": int(len(y_true)),
        "n_episodes": int(len(ep_df)),
        "win_mae": win_mae,
        "win_rmse": win_rmse,
        "win_r2": win_r2,
        "win_spearman": win_rho,
        "ep_mae": ep_mae,
        "ep_rmse": ep_rmse,
        "ep_r2": ep_r2,
        "ep_spearman": ep_rho,
        "mean_within_p_spearman": float(np.mean(wp_rhos)) if wp_rhos else 0.0,
    }


def eval_classification(y_true, probs, episode_ids, n_classes: int = 3) -> dict:
    labels_idx = list(range(n_classes))
    y_pred = probs.argmax(axis=1)
    rec = recall_score(y_true, y_pred, labels=labels_idx, average=None, zero_division=0)
    res = {
        "n_windows": int(len(y_true)),
        "win_accuracy": float(accuracy_score(y_true, y_pred)),
        "win_bal_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "win_macro_f1": float(f1_score(y_true, y_pred, labels=labels_idx, average="macro", zero_division=0)),
        "win_min_recall": float(rec.min()),
    }
    if n_classes == 2:
        res["win_auc"] = float(roc_auc_score(y_true, probs[:, 1]))
    else:
        res["win_recall_low"] = float(rec[0])
        res["win_recall_med"] = float(rec[1])
        res["win_recall_high"] = float(rec[2])

    cols_p = {f"p{i}": probs[:, i] for i in labels_idx}
    ep_df = pd.DataFrame({"ep": episode_ids, "y": y_true, **cols_p}).groupby("ep", as_index=False).mean()
    ep_df["y"] = ep_df["y"].round().astype(int)
    ep_probs = ep_df[[f"p{i}" for i in labels_idx]].to_numpy()
    ep_pred = ep_probs.argmax(axis=1)
    ep_true = ep_df["y"].to_numpy()
    ep_rec = recall_score(ep_true, ep_pred, labels=labels_idx, average=None, zero_division=0)
    res.update({
        "n_episodes": int(len(ep_df)),
        "ep_accuracy": float(accuracy_score(ep_true, ep_pred)),
        "ep_bal_acc": float(balanced_accuracy_score(ep_true, ep_pred)),
        "ep_macro_f1": float(f1_score(ep_true, ep_pred, labels=labels_idx, average="macro", zero_division=0)),
        "ep_min_recall": float(ep_rec.min()),
        "ep_all_recalled": bool((ep_rec > 0).all()),
    })
    if n_classes == 2:
        res["ep_auc"] = float(roc_auc_score(ep_true, ep_probs[:, 1]))
    else:
        res["ep_recall_low"] = float(ep_rec[0])
        res["ep_recall_med"] = float(ep_rec[1])
        res["ep_recall_high"] = float(ep_rec[2])
    return res


In [4]:
# Cell 4: Track A — Regresi Kesulitan 1–10 (5-Fold Participant CV di Dev -> Kunci -> Test)
dev_df = df[df["split"] == "Dev"].reset_index(drop=True)
test_df = df[df["split"] == "Test"].reset_index(drop=True)
gkf = GroupKFold(n_splits=5)
dev_splits = list(gkf.split(dev_df, dev_df["difficulty_rating"], dev_df["participant_id"]))

NORMS = ["global_robust", "causal_session_robust", "unsupervised_session_robust"]
reg_specs = [("DummyMean", "core10", "global_robust", "uniform", "window")]
for norm in NORMS:
    for m in ["Ridge", "ExtraTreesReg", "HistGBReg", "PyTorchMLPReg"]:
        reg_specs.append((m, "core9", norm, "episode_equalized_clipped", "window"))
for feat in ["core10", "core9_head4", "core9_head4_base"]:
    for norm in ["global_robust", "causal_session_robust", "unsupervised_session_robust"]:
        for m in ["Ridge", "ExtraTreesReg", "HistGBReg"]:
            reg_specs.append((m, feat, norm, "episode_equalized_clipped", "window"))
for feat in ["core9", "core9_head4_base"]:
    for norm in ["global_robust", "unsupervised_session_robust"]:
        for m in ["Ridge", "ExtraTreesReg"]:
            reg_specs.append((m, feat, norm, "uniform", "episode_summary"))

t0 = time.time()
dev_fold_cache = {}

def get_dev_fold_xy(fold_i, tr_i, va_i, feat, norm, gran):
    key = (fold_i, feat, norm, gran)
    if key not in dev_fold_cache:
        f_tr, f_va = dev_df.iloc[tr_i], dev_df.iloc[va_i]
        cols = FEATURE_PROFILES[feat]
        prep = V4SensorPreprocessor(cols, norm).fit(f_tr)
        if gran == "window":
            dev_fold_cache[key] = (prep.transform(f_tr), prep.transform(f_va))
        else:
            dev_fold_cache[key] = (make_v4_episode_summary(f_tr, cols, prep), make_v4_episode_summary(f_va, cols, prep))
    return dev_fold_cache[key]

reg_rows = []
for m_name, feat, norm, w_mode, gran in reg_specs:
    cid = f"{gran}|{m_name}|{feat}|{norm}|{w_mode}"
    oof_y, oof_p, oof_ep, oof_pid = [], [], [], []
    for fold_i, (tr_i, va_i) in enumerate(dev_splits):
        f_tr, f_va = dev_df.iloc[tr_i], dev_df.iloc[va_i]
        if gran == "window":
            X_tr, X_va = get_dev_fold_xy(fold_i, tr_i, va_i, feat, norm, gran)
            y_tr = f_tr["difficulty_rating"].to_numpy(float)
            y_va = f_va["difficulty_rating"].to_numpy(float)
            w_tr = compute_v4_weights(f_tr, None, w_mode)
            pred = fit_predict_reg(m_name, X_tr, y_tr, w_tr, f_tr["participant_id"].to_numpy(), X_va)
            oof_y.append(y_va); oof_p.append(pred)
            oof_ep.append(f_va["episode_id"].to_numpy()); oof_pid.append(f_va["participant_id"].to_numpy())
        else:
            tr_ep, va_ep = get_dev_fold_xy(fold_i, tr_i, va_i, feat, norm, gran)
            sc = [c for c in tr_ep.columns if c not in ("episode_id", "participant_id", "difficulty_rating", "tier")]
            pred = fit_predict_reg(
                m_name, tr_ep[sc].to_numpy(float), tr_ep["difficulty_rating"].to_numpy(float),
                np.ones(len(tr_ep)), tr_ep["participant_id"].to_numpy(), va_ep[sc].to_numpy(float)
            )
            oof_y.append(va_ep["difficulty_rating"].to_numpy(float)); oof_p.append(pred)
            oof_ep.append(va_ep["episode_id"].to_numpy()); oof_pid.append(va_ep["participant_id"].to_numpy())
    met = eval_regression(np.concatenate(oof_y), np.concatenate(oof_p), np.concatenate(oof_ep), np.concatenate(oof_pid))
    reg_rows.append({"candidate_id": cid, "granularity": gran, "model": m_name, "features": feat, "normalization": norm, "weighting": w_mode, **met})

reg_lb = pd.DataFrame(reg_rows).sort_values(["ep_mae", "win_mae", "ep_spearman"], ascending=[True, True, False]).reset_index(drop=True)
reg_lb.to_csv(OUTPUT_DIR / "trackA_regression_cv_leaderboard.csv", index=False)

# Kunci pemenang operasional per-window (global_robust / causal_session_robust, bukan Dummy;
# cabang episode_summary dipisahkan sebagai diagnostik karena iqr/slope=0 pada episode pendek membocorkan durasi)
reg_elig = reg_lb[(reg_lb["model"] != "DummyMean") & (reg_lb["granularity"] == "window") & (reg_lb["normalization"].isin(["global_robust", "causal_session_robust"]))]
reg_winner = reg_elig.iloc[0]
reg_dummy = reg_lb[reg_lb["model"] == "DummyMean"].iloc[0]
reg_unsup = reg_lb[(reg_lb["model"] != "DummyMean") & (reg_lb["granularity"] == "window") & (reg_lb["normalization"] == "unsupervised_session_robust")].iloc[0]

def eval_reg_on_test(spec):
    cols = FEATURE_PROFILES[spec["features"]]
    prep = V4SensorPreprocessor(cols, spec["normalization"]).fit(dev_df)
    if spec["granularity"] == "window":
        pred = fit_predict_reg(
            spec["model"], prep.transform(dev_df), dev_df["difficulty_rating"].to_numpy(float),
            compute_v4_weights(dev_df, None, spec["weighting"]), dev_df["participant_id"].to_numpy(), prep.transform(test_df)
        )
        met = eval_regression(test_df["difficulty_rating"].to_numpy(float), pred, test_df["episode_id"].to_numpy(), test_df["participant_id"].to_numpy())
        p_df = test_df[["window_id", "participant_id", "episode_id", "difficulty_rating"]].copy()
        p_df["pred_rating"] = pred
        return met, p_df
    else:
        tr_ep = make_v4_episode_summary(dev_df, cols, prep)
        te_ep = make_v4_episode_summary(test_df, cols, prep)
        sc = [c for c in tr_ep.columns if c not in ("episode_id", "participant_id", "difficulty_rating", "tier")]
        pred = fit_predict_reg(
            spec["model"], tr_ep[sc].to_numpy(float), tr_ep["difficulty_rating"].to_numpy(float),
            np.ones(len(tr_ep)), tr_ep["participant_id"].to_numpy(), te_ep[sc].to_numpy(float)
        )
        met = eval_regression(te_ep["difficulty_rating"].to_numpy(float), pred, te_ep["episode_id"].to_numpy(), te_ep["participant_id"].to_numpy())
        p_df = te_ep[["episode_id", "participant_id", "difficulty_rating"]].copy()
        p_df["pred_rating"] = pred
        return met, p_df

reg_test_win, reg_test_pred_df = eval_reg_on_test(reg_winner)
reg_test_dum, _ = eval_reg_on_test(reg_dummy)
reg_test_unsup, _ = eval_reg_on_test(reg_unsup)
reg_test_pred_df.to_csv(OUTPUT_DIR / "trackA_test_predictions.csv", index=False)

print(f"Track A selesai ({len(reg_specs)} kandidat, {time.time()-t0:.1f}s).")
print("Top 6 Dev OOF Regression:")
print(reg_lb[["candidate_id", "win_mae", "win_r2", "ep_mae", "ep_r2", "ep_spearman"]].head(6).to_string(index=False))
print(f"\nLocked Winner Test ({reg_winner['candidate_id']}): win_MAE={reg_test_win['win_mae']:.3f}, win_R2={reg_test_win['win_r2']:+.3f} | ep_MAE={reg_test_win['ep_mae']:.3f}, ep_R2={reg_test_win['ep_r2']:+.3f}, ep_rho={reg_test_win['ep_spearman']:+.3f}")
print(f"DummyMean Test: win_MAE={reg_test_dum['win_mae']:.3f}, ep_MAE={reg_test_dum['ep_mae']:.3f}")
print(f"Unsup-Ref Test ({reg_unsup['candidate_id']}): ep_MAE={reg_test_unsup['ep_mae']:.3f}, ep_R2={reg_test_unsup['ep_r2']:+.3f}, ep_rho={reg_test_unsup['ep_spearman']:+.3f}")


Track A selesai (48 kandidat, 89.6s).
Top 6 Dev OOF Regression:
                                                                      candidate_id  win_mae    win_r2   ep_mae     ep_r2  ep_spearman
episode_summary|ExtraTreesReg|core9_head4_base|unsupervised_session_robust|uniform 2.105465  0.127804 2.105465  0.127804     0.227297
           episode_summary|ExtraTreesReg|core9|unsupervised_session_robust|uniform 2.140471  0.089558 2.140471  0.089558     0.216290
         window|Ridge|core10|unsupervised_session_robust|episode_equalized_clipped 2.919376 -0.392251 2.295058 -0.061435    -0.065932
          window|Ridge|core9|unsupervised_session_robust|episode_equalized_clipped 2.919464 -0.392249 2.295497 -0.061703    -0.067746
              episode_summary|ExtraTreesReg|core9_head4_base|global_robust|uniform 2.304594  0.030859 2.304594  0.030859     0.272978
  window|PyTorchMLPReg|core9|unsupervised_session_robust|episode_equalized_clipped 3.212491 -0.708426 2.329023 -0.156084    -0.26954

In [5]:
# Cell 5: Track B — Klasifikasi 3-Kelas Perceived Load (1–3 Low, 4–6 Med, 7–10 High)
clf_specs = [("Dummy", "core10", "global_robust", "uniform", "window")]
for norm in NORMS:
    for m in ["LogisticRegression", "ExtraTrees", "HistGradientBoosting", "PyTorchMLP"]:
        for wm in (["episode_equalized_clipped", "class_balanced"] if m != "PyTorchMLP" else ["episode_equalized_clipped"]):
            clf_specs.append((m, "core9", norm, wm, "window"))
for feat in ["core10", "core9_head4", "core9_head4_base"]:
    for norm in ["global_robust", "causal_session_robust", "unsupervised_session_robust"]:
        for m in ["LogisticRegression", "ExtraTrees", "HistGradientBoosting"]:
            clf_specs.append((m, feat, norm, "episode_equalized_clipped", "window"))
for feat in ["core9", "core9_head4_base"]:
    for norm in ["global_robust", "unsupervised_session_robust"]:
        for m in ["LogisticRegression", "ExtraTrees"]:
            clf_specs.append((m, feat, norm, "class_balanced", "episode_summary"))

t0 = time.time()
clf_rows = []
for m_name, feat, norm, w_mode, gran in clf_specs:
    cid = f"{gran}|{m_name}|{feat}|{norm}|{w_mode}"
    oof_y, oof_p, oof_ep = [], [], []
    for fold_i, (tr_i, va_i) in enumerate(dev_splits):
        f_tr, f_va = dev_df.iloc[tr_i], dev_df.iloc[va_i]
        if gran == "window":
            X_tr, X_va = get_dev_fold_xy(fold_i, tr_i, va_i, feat, norm, gran)
            y_tr = f_tr["tier"].map(TIER_TO_IDX).to_numpy()
            y_va = f_va["tier"].map(TIER_TO_IDX).to_numpy()
            w_tr = compute_v4_weights(f_tr, "tier", w_mode)
            probs = fit_predict_clf(m_name, X_tr, y_tr, w_tr, f_tr["participant_id"].to_numpy(), X_va, n_classes=3)
            oof_y.append(y_va); oof_p.append(probs); oof_ep.append(f_va["episode_id"].to_numpy())
        else:
            tr_ep, va_ep = get_dev_fold_xy(fold_i, tr_i, va_i, feat, norm, gran)
            sc = [c for c in tr_ep.columns if c not in ("episode_id", "participant_id", "difficulty_rating", "tier")]
            y_tr = tr_ep["tier"].map(TIER_TO_IDX).to_numpy()
            y_va = va_ep["tier"].map(TIER_TO_IDX).to_numpy()
            vc = pd.Series(y_tr).value_counts()
            w_ep = np.array([len(y_tr) / (len(vc) * vc[y]) for y in y_tr], dtype=float)
            probs = fit_predict_clf(m_name, tr_ep[sc].to_numpy(float), y_tr, w_ep, tr_ep["participant_id"].to_numpy(), va_ep[sc].to_numpy(float), n_classes=3)
            oof_y.append(y_va); oof_p.append(probs); oof_ep.append(va_ep["episode_id"].to_numpy())
    met = eval_classification(np.concatenate(oof_y), np.vstack(oof_p), np.concatenate(oof_ep), n_classes=3)
    clf_rows.append({"candidate_id": cid, "granularity": gran, "model": m_name, "features": feat, "normalization": norm, "weighting": w_mode, **met})

clf_lb = pd.DataFrame(clf_rows).sort_values(
    ["ep_all_recalled", "ep_macro_f1", "win_macro_f1", "win_bal_acc"], ascending=False
).reset_index(drop=True)
clf_lb.to_csv(OUTPUT_DIR / "trackB_3class_cv_leaderboard.csv", index=False)

clf_elig = clf_lb[(clf_lb["model"] != "Dummy") & (clf_lb["granularity"] == "window") & (clf_lb["normalization"].isin(["global_robust", "causal_session_robust"]))]
clf_winner = clf_elig.iloc[0]
clf_dummy = clf_lb[clf_lb["model"] == "Dummy"].iloc[0]
clf_unsup = clf_lb[(clf_lb["model"] != "Dummy") & (clf_lb["granularity"] == "window") & (clf_lb["normalization"] == "unsupervised_session_robust")].iloc[0]

def eval_clf_on_test(spec):
    cols = FEATURE_PROFILES[spec["features"]]
    prep = V4SensorPreprocessor(cols, spec["normalization"]).fit(dev_df)
    if spec["granularity"] == "window":
        y_tr = dev_df["tier"].map(TIER_TO_IDX).to_numpy()
        y_te = test_df["tier"].map(TIER_TO_IDX).to_numpy()
        probs = fit_predict_clf(
            spec["model"], prep.transform(dev_df), y_tr,
            compute_v4_weights(dev_df, "tier", spec["weighting"]),
            dev_df["participant_id"].to_numpy(), prep.transform(test_df), n_classes=3
        )
        met = eval_classification(y_te, probs, test_df["episode_id"].to_numpy(), n_classes=3)
        p_df = test_df[["window_id", "participant_id", "episode_id", "difficulty_rating", "tier"]].copy()
        p_df["pred_tier"] = [TIERS[i] for i in probs.argmax(axis=1)]
        for i, t in enumerate(TIERS):
            p_df[f"prob_{t}"] = probs[:, i]
        return met, p_df
    else:
        tr_ep = make_v4_episode_summary(dev_df, cols, prep)
        te_ep = make_v4_episode_summary(test_df, cols, prep)
        sc = [c for c in tr_ep.columns if c not in ("episode_id", "participant_id", "difficulty_rating", "tier")]
        y_tr = tr_ep["tier"].map(TIER_TO_IDX).to_numpy()
        y_te = te_ep["tier"].map(TIER_TO_IDX).to_numpy()
        vc = pd.Series(y_tr).value_counts()
        w_ep = np.array([len(y_tr) / (len(vc) * vc[y]) for y in y_tr], dtype=float)
        probs = fit_predict_clf(spec["model"], tr_ep[sc].to_numpy(float), y_tr, w_ep, tr_ep["participant_id"].to_numpy(), te_ep[sc].to_numpy(float), n_classes=3)
        met = eval_classification(y_te, probs, te_ep["episode_id"].to_numpy(), n_classes=3)
        p_df = te_ep[["episode_id", "participant_id", "difficulty_rating", "tier"]].copy()
        p_df["pred_tier"] = [TIERS[i] for i in probs.argmax(axis=1)]
        for i, t in enumerate(TIERS):
            p_df[f"prob_{t}"] = probs[:, i]
        return met, p_df

clf_test_win, clf_test_pred_df = eval_clf_on_test(clf_winner)
clf_test_dum, _ = eval_clf_on_test(clf_dummy)
clf_test_unsup, _ = eval_clf_on_test(clf_unsup)
clf_test_pred_df.to_csv(OUTPUT_DIR / "trackB_test_predictions.csv", index=False)

print(f"Track B selesai ({len(clf_specs)} kandidat, {time.time()-t0:.1f}s).")
print("Top 6 Dev OOF 3-Class:")
print(clf_lb[["candidate_id", "win_accuracy", "win_macro_f1", "ep_accuracy", "ep_macro_f1", "ep_all_recalled"]].head(6).to_string(index=False))
print(f"\nLocked Winner Test ({clf_winner['candidate_id']}): win_Acc={clf_test_win['win_accuracy']:.2%}, win_mF1={clf_test_win['win_macro_f1']:.2%} | ep_Acc={clf_test_win['ep_accuracy']:.2%}, ep_mF1={clf_test_win['ep_macro_f1']:.2%}")
print(f"Dummy Test: win_Acc={clf_test_dum['win_accuracy']:.2%}, win_mF1={clf_test_dum['win_macro_f1']:.2%} | ep_Acc={clf_test_dum['ep_accuracy']:.2%}, ep_mF1={clf_test_dum['ep_macro_f1']:.2%}")
print(f"Unsup-Ref Test ({clf_unsup['candidate_id']}): win_mF1={clf_test_unsup['win_macro_f1']:.2%} | ep_mF1={clf_test_unsup['ep_macro_f1']:.2%}")


Track B selesai (57 kandidat, 111.8s).
Top 6 Dev OOF 3-Class:
                                                                          candidate_id  win_accuracy  win_macro_f1  ep_accuracy  ep_macro_f1  ep_all_recalled
episode_summary|ExtraTrees|core9_head4_base|unsupervised_session_robust|class_balanced      0.464646      0.424603     0.464646     0.424603             True
      window|LogisticRegression|core10|causal_session_robust|episode_equalized_clipped      0.377259      0.361455     0.434343     0.417503             True
    window|LogisticRegression|core9_head4_base|global_robust|episode_equalized_clipped      0.469880      0.420529     0.444444     0.395052             True
                         episode_summary|ExtraTrees|core9|global_robust|class_balanced      0.434343      0.380017     0.434343     0.380017             True
              episode_summary|ExtraTrees|core9_head4_base|global_robust|class_balanced      0.424242      0.377451     0.424242     0.377451        

In [6]:
# Cell 6: Track C1 (Biner Ekstrem 1–3 vs 7–10) & Track C2 (Diagnostik Dalam-Peserta LOEO)
# --- Track C1: Cross-Participant Extreme Binary (1–3 Low vs 7–10 High) ---
dev_bin = dev_df[dev_df["tier"].isin(["Low", "High"])].reset_index(drop=True)
test_bin = test_df[test_df["tier"].isin(["Low", "High"])].reset_index(drop=True)
bin_gkf = GroupKFold(n_splits=5)
bin_splits = list(bin_gkf.split(dev_bin, (dev_bin["tier"] == "High").astype(int), dev_bin["participant_id"]))

bin_rows = []
for feat in ["core9", "core9_head4", "core9_head4_base"]:
    cols = FEATURE_PROFILES[feat]
    for norm in NORMS:
        for m in ["LogisticRegression", "ExtraTrees", "HistGradientBoosting", "PyTorchMLP"]:
            oof_y, oof_p, oof_ep = [], [], []
            for tr_i, va_i in bin_splits:
                f_tr, f_va = dev_bin.iloc[tr_i], dev_bin.iloc[va_i]
                prep = V4SensorPreprocessor(cols, norm).fit(f_tr)
                y_tr = (f_tr["tier"] == "High").astype(int).to_numpy()
                y_va = (f_va["tier"] == "High").astype(int).to_numpy()
                w_tr = compute_v4_weights(f_tr, "tier", "episode_equalized_clipped")
                probs = fit_predict_clf(m, prep.transform(f_tr), y_tr, w_tr, f_tr["participant_id"].to_numpy(), prep.transform(f_va), n_classes=2)
                oof_y.append(y_va); oof_p.append(probs); oof_ep.append(f_va["episode_id"].to_numpy())
            met = eval_classification(np.concatenate(oof_y), np.vstack(oof_p), np.concatenate(oof_ep), n_classes=2)
            bin_rows.append({"candidate_id": f"{m}|{feat}|{norm}", "model": m, "features": feat, "normalization": norm, **met})

bin_lb = pd.DataFrame(bin_rows).sort_values(["ep_auc", "win_auc", "ep_macro_f1"], ascending=False).reset_index(drop=True)
bin_lb.to_csv(OUTPUT_DIR / "trackC1_binary_cv_leaderboard.csv", index=False)

bin_elig = bin_lb[bin_lb["normalization"].isin(["global_robust", "causal_session_robust"])]
bin_winner = bin_elig.iloc[0]
# Evaluasi 1x pada Test Biner
cols_b = FEATURE_PROFILES[bin_winner["features"]]
prep_b = V4SensorPreprocessor(cols_b, bin_winner["normalization"]).fit(dev_bin)
probs_b = fit_predict_clf(
    bin_winner["model"], prep_b.transform(dev_bin), (dev_bin["tier"] == "High").astype(int).to_numpy(),
    compute_v4_weights(dev_bin, "tier", "episode_equalized_clipped"),
    dev_bin["participant_id"].to_numpy(), prep_b.transform(test_bin), n_classes=2
)
bin_test_met = eval_classification((test_bin["tier"] == "High").astype(int).to_numpy(), probs_b, test_bin["episode_id"].to_numpy(), n_classes=2)

print("Top 5 Track C1 (Extreme Binary 1–3 vs 7–10) Dev OOF:")
print(bin_lb[["candidate_id", "win_auc", "win_bal_acc", "ep_auc", "ep_bal_acc"]].head(5).to_string(index=False))
print(f"Locked Binary Winner Test ({bin_winner['candidate_id']}): win_AUC={bin_test_met['win_auc']:.3f}, win_BalAcc={bin_test_met['win_bal_acc']:.2%} | ep_AUC={bin_test_met['ep_auc']:.3f}, ep_BalAcc={bin_test_met['ep_bal_acc']:.2%}")

# --- Track C2: Diagnostik Kalibrasi Dalam-Peserta (Leave-One-Episode-Out per Peserta) ---
# Menguji apakah fitur fisiologis dapat memprediksi rating di DALAM orang yang sama (bebas overlap episode).
wp_rows = []
diag_cols = FEATURE_PROFILES["core9_head4"]
for pid, g in df.groupby("participant_id"):
    eps = sorted(g["episode_id"].unique())
    if len(eps) < 4 or g["difficulty_rating"].nunique() < 2:
        continue
    # Leave-One-Episode-Out regression (Ridge & ExtraTrees) dalam 1 peserta
    for m_name in ["Ridge", "ExtraTreesReg"]:
        ep_trues, ep_preds, win_trues, win_preds = [], [], [], []
        for hold_ep in eps:
            tr_p = g[g["episode_id"] != hold_ep]
            va_p = g[g["episode_id"] == hold_ep]
            prep_p = V4SensorPreprocessor(diag_cols, "global_robust").fit(tr_p)
            pred = fit_predict_reg(
                m_name, prep_p.transform(tr_p), tr_p["difficulty_rating"].to_numpy(float),
                np.ones(len(tr_p)), tr_p["participant_id"].to_numpy(), prep_p.transform(va_p)
            )
            win_trues.extend(va_p["difficulty_rating"].to_numpy(float))
            win_preds.extend(pred)
            ep_trues.append(float(va_p["difficulty_rating"].iloc[0]))
            ep_preds.append(float(np.mean(pred)))
        ep_trues, ep_preds = np.array(ep_trues), np.array(ep_preds)
        dummy_mae = float(np.mean(np.abs(ep_trues - np.mean(ep_trues))))
        ep_mae = float(mean_absolute_error(ep_trues, ep_preds))
        ep_rho = float(stats.spearmanr(ep_trues, ep_preds).statistic) if np.std(ep_preds) > 1e-9 else 0.0
        # Cek binary AUC jika peserta punya >=2 Low (1-3) dan >=2 High (7-10)
        ep_tiers = [g[g["episode_id"] == e]["tier"].iloc[0] for e in eps]
        low_high_idx = [i for i, t in enumerate(ep_tiers) if t in ("Low", "High")]
        bin_auc = np.nan
        if len(low_high_idx) >= 4:
            yb = np.array([1 if ep_tiers[i] == "High" else 0 for i in low_high_idx])
            if yb.sum() >= 2 and (1 - yb).sum() >= 2:
                bin_auc = float(roc_auc_score(yb, ep_preds[low_high_idx]))
        wp_rows.append({
            "participant_id": pid, "model": m_name, "n_episodes": len(eps), "n_windows": len(g),
            "ep_dummy_mae": dummy_mae, "ep_loeo_mae": ep_mae, "mae_gain": dummy_mae - ep_mae,
            "ep_loeo_spearman": ep_rho, "ep_binary_auc_if_eligible": bin_auc,
        })

wp_df = pd.DataFrame(wp_rows)
wp_df.to_csv(OUTPUT_DIR / "trackC2_within_participant_loeo.csv", index=False)
print("\nTrack C2 — Within-Participant Leave-One-Episode-Out Summary (Core9+Head4):")
for m_name, sub_w in wp_df.groupby("model"):
    print(
        f"  {m_name:14s} (n_people={len(sub_w)}): mean ep_MAE={sub_w['ep_loeo_mae'].mean():.2f} (dummy={sub_w['ep_dummy_mae'].mean():.2f}), "
        f"median ep_Spearman={sub_w['ep_loeo_spearman'].median():+.3f} (pos={(sub_w['ep_loeo_spearman']>0).sum()}/{len(sub_w)}), "
        f"mean bin_AUC(n={sub_w['ep_binary_auc_if_eligible'].notna().sum()})={sub_w['ep_binary_auc_if_eligible'].mean():.3f}"
    )


Top 5 Track C1 (Extreme Binary 1–3 vs 7–10) Dev OOF:
                                             candidate_id  win_auc  win_bal_acc   ep_auc  ep_bal_acc
        LogisticRegression|core9_head4_base|global_robust 0.699765     0.646910 0.739316    0.725427
LogisticRegression|core9_head4_base|causal_session_robust 0.606175     0.577632 0.736111    0.677350
     LogisticRegression|core9_head4|causal_session_robust 0.604134     0.576479 0.723291    0.677350
           LogisticRegression|core9|causal_session_robust 0.618692     0.597960 0.697650    0.677350
                   LogisticRegression|core9|global_robust 0.681762     0.607282 0.696581    0.631410
Locked Binary Winner Test (LogisticRegression|core9_head4_base|global_robust): win_AUC=0.562, win_BalAcc=53.96% | ep_AUC=0.714, ep_BalAcc=62.14%

Track C2 — Within-Participant Leave-One-Episode-Out Summary (Core9+Head4):
  ExtraTreesReg  (n_people=13): mean ep_MAE=2.06 (dummy=1.67), median ep_Spearman=-0.237 (pos=3/13), mean bin_AUC(n=3)=0

In [7]:
# Cell 7: Simpan Ringkasan JSON, Visualisasi, dan RESULTS.md
# 1. Visualisasi Track A (Episode Scatter) & Track B (Test Confusion Matrix) & Track C2 (Within-Participant Spearman)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel 1: Track A Episode True vs Pred on Held-Out Test
if "window_id" in reg_test_pred_df.columns:
    ep_sc = reg_test_pred_df.groupby(["episode_id", "participant_id"], as_index=False).agg(
        y=("difficulty_rating", "first"), p=("pred_rating", "mean")
    )
else:
    ep_sc = reg_test_pred_df.rename(columns={"difficulty_rating": "y", "pred_rating": "p"})
axes[0].scatter(ep_sc["y"], ep_sc["p"], alpha=0.8, color="#2563eb", edgecolors="white", s=55)
axes[0].plot([1, 10], [1, 10], "r--", linewidth=1, label="Ideal (y=x)")
axes[0].set_xlim(0.5, 10.5); axes[0].set_ylim(0.5, 10.5)
axes[0].set_xlabel("True Difficulty Rating"); axes[0].set_ylabel("Predicted Rating (Episode Mean)")
axes[0].set_title(f"Track A Test (ep MAE={reg_test_win['ep_mae']:.2f}, rho={reg_test_win['ep_spearman']:+.2f})")
axes[0].legend(fontsize=8)

# Panel 2: Track B Test Episode Confusion Matrix
if "window_id" in clf_test_pred_df.columns:
    ep_cm_df = clf_test_pred_df.groupby("episode_id", as_index=False).agg(
        tier=("tier", "first"), p0=("prob_Low", "mean"), p1=("prob_Med", "mean"), p2=("prob_High", "mean")
    )
    ep_cm_df["pred_tier"] = [TIERS[i] for i in ep_cm_df[["p0", "p1", "p2"]].to_numpy().argmax(axis=1)]
else:
    ep_cm_df = clf_test_pred_df.copy()
cm_b = confusion_matrix(ep_cm_df["tier"].map(TIER_TO_IDX), ep_cm_df["pred_tier"].map(TIER_TO_IDX), labels=[0, 1, 2])
axes[1].imshow(cm_b, cmap="Blues")
axes[1].set_xticks([0, 1, 2]); axes[1].set_xticklabels(TIERS)
axes[1].set_yticks([0, 1, 2]); axes[1].set_yticklabels(TIERS)
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")
axes[1].set_title(f"Track B Test Episode CM (mF1={clf_test_win['ep_macro_f1']:.1%})")
for i in range(3):
    for j in range(3):
        axes[1].text(j, i, str(cm_b[i, j]), ha="center", va="center", color="white" if cm_b[i, j] > cm_b.max()/2 else "black", fontsize=11)

# Panel 3: Track C2 Within-Participant LOEO Spearman per person
wp_et = wp_df[wp_df["model"] == "ExtraTreesReg"].sort_values("ep_loeo_spearman")
colors = ["#10b981" if v > 0 else "#ef4444" for v in wp_et["ep_loeo_spearman"]]
axes[2].barh(wp_et["participant_id"], wp_et["ep_loeo_spearman"], color=colors, alpha=0.85)
axes[2].axvline(0.0, color="black", linewidth=1)
axes[2].set_xlim(-1.0, 1.0)
axes[2].set_xlabel("LOEO Episode Spearman rho")
axes[2].set_title("Track C2: Within-Participant LOEO (ExtraTrees)")
axes[2].tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig(FIG_DIR / "v4_experiment_overview.png", dpi=150)
plt.close(fig)

# 2. Visualisasi Ablasi Fitur & Normalisasi Track B (3-Class Dev OOF)
fig, ax = plt.subplots(figsize=(9, 4.5))
sub_b = clf_lb[(clf_lb["granularity"] == "window") & (clf_lb["model"] != "Dummy") & (clf_lb["weighting"] == "episode_equalized_clipped")]
piv = sub_b.pivot_table(index="features", columns="normalization", values="ep_macro_f1", aggfunc="mean")
piv.plot(kind="bar", ax=ax, rot=0)
ax.axhline(1/3, color="red", linestyle="--", linewidth=1, label="Random (33.3%)")
ax.set_ylabel("Mean Dev OOF Episode Macro-F1")
ax.set_title("V4 Track B (3-Class): Ablasi Profil Fitur vs Normalisasi (Dev OOF)")
ax.set_ylim(0, 0.7)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "v4_ablation_trackB.png", dpi=150)
plt.close(fig)

# Simpan test_evaluation.json dan run.json
def clean_dict(d):
    out = {}
    for k, v in d.items():
        if isinstance(v, (np.floating, float)):
            out[k] = None if np.isnan(v) else float(v)
        elif isinstance(v, (np.integer, int)):
            out[k] = int(v)
        elif isinstance(v, (np.bool_, bool)):
            out[k] = bool(v)
        else:
            out[k] = v
    return out

wp_summary = {
    m: {
        "n_participants": int(len(g)),
        "mean_ep_dummy_mae": float(g["ep_dummy_mae"].mean()),
        "mean_ep_loeo_mae": float(g["ep_loeo_mae"].mean()),
        "median_ep_spearman": float(g["ep_loeo_spearman"].median()),
        "positive_spearman_count": int((g["ep_loeo_spearman"] > 0).sum()),
        "mean_binary_auc": float(g["ep_binary_auc_if_eligible"].mean()) if g["ep_binary_auc_if_eligible"].notna().any() else None,
    }
    for m, g in wp_df.groupby("model")
}

eval_payload = {
    "test_participants": sorted(test_people),
    "trackA_regression": {
        "locked_winner_id": reg_winner["candidate_id"],
        "dev_oof": clean_dict(reg_winner.to_dict()),
        "test_locked": reg_test_win,
        "test_dummy": reg_test_dum,
        "test_unsup_ref": {"candidate_id": reg_unsup["candidate_id"], **reg_test_unsup},
    },
    "trackB_3class": {
        "locked_winner_id": clf_winner["candidate_id"],
        "dev_oof": clean_dict(clf_winner.to_dict()),
        "test_locked": clf_test_win,
        "test_dummy": clf_test_dum,
        "test_unsup_ref": {"candidate_id": clf_unsup["candidate_id"], **clf_test_unsup},
    },
    "trackC1_binary": {
        "locked_winner_id": bin_winner["candidate_id"],
        "dev_oof": clean_dict(bin_winner.to_dict()),
        "test_locked": bin_test_met,
    },
    "trackC2_within_participant_loeo": wp_summary,
}
(OUTPUT_DIR / "test_evaluation.json").write_text(json.dumps(eval_payload, indent=2), encoding="utf-8")

run_meta = {
    "experiment": "batch-06/exp02_v4_self_reported_rating",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "device": DEVICE,
    "seed": SEED,
    "dataset_sha256": file_hashes,
    "dev_participants": sorted(set(people) - test_people),
    "test_participants": sorted(test_people),
}
(OUTPUT_DIR / "run.json").write_text(json.dumps(run_meta, indent=2), encoding="utf-8")

# Tulis RESULTS.md
md = [
    "# Hasil Eksperimen Batch 06 — Exp02: `v4_self_reported_rating`",
    "",
    f"- **Tanggal run (UTC):** {run_meta['generated_at_utc']}",
    f"- **Pembagian peserta:** 13 Development / 4 Held-Out Test ({', '.join(sorted(test_people))})",
    "",
    "## Track A: Regresi Rating Kesulitan (1–10)",
    f"- **Pemenang terkunci (Dev OOF):** `{reg_winner['candidate_id']}`",
    "",
    "| Evaluasi | Window MAE | Window $R^2$ | Episode MAE | Episode $R^2$ | Episode Spearman $\rho$ |",
    "|---|---:|---:|---:|---:|---:|",
    f"| Dev 5-Fold OOF (13 org) | {reg_winner['win_mae']:.3f} | {reg_winner['win_r2']:+.3f} | {reg_winner['ep_mae']:.3f} | {reg_winner['ep_r2']:+.3f} | {reg_winner['ep_spearman']:+.3f} |",
    f"| **Held-Out Test (4 org)** | **{reg_test_win['win_mae']:.3f}** | **{reg_test_win['win_r2']:+.3f}** | **{reg_test_win['ep_mae']:.3f}** | **{reg_test_win['ep_r2']:+.3f}** | **{reg_test_win['ep_spearman']:+.3f}** |",
    f"| Dummy Mean (Test) | {reg_test_dum['win_mae']:.3f} | {reg_test_dum['win_r2']:+.3f} | {reg_test_dum['ep_mae']:.3f} | {reg_test_dum['ep_r2']:+.3f} | {reg_test_dum['ep_spearman']:+.3f} |",
    f"| Unsup Session-Norm Ref (Test) | {reg_test_unsup['win_mae']:.3f} | {reg_test_unsup['win_r2']:+.3f} | {reg_test_unsup['ep_mae']:.3f} | {reg_test_unsup['ep_r2']:+.3f} | {reg_test_unsup['ep_spearman']:+.3f} |",
    "",
    "## Track B: Klasifikasi 3-Kelas (`1–3` Low, `4–6` Med, `7–10` High)",
    f"- **Pemenang terkunci (Dev OOF):** `{clf_winner['candidate_id']}`",
    "",
    "| Evaluasi | Window Acc | Window BalAcc | Window Macro-F1 | Episode Acc | Episode BalAcc | Episode Macro-F1 |",
    "|---|---:|---:|---:|---:|---:|---:|",
    f"| Dev 5-Fold OOF (13 org) | {clf_winner['win_accuracy']:.2%} | {clf_winner['win_bal_acc']:.2%} | {clf_winner['win_macro_f1']:.2%} | {clf_winner['ep_accuracy']:.2%} | {clf_winner['ep_bal_acc']:.2%} | {clf_winner['ep_macro_f1']:.2%} |",
    f"| **Held-Out Test (4 org)** | **{clf_test_win['win_accuracy']:.2%}** | **{clf_test_win['win_bal_acc']:.2%}** | **{clf_test_win['win_macro_f1']:.2%}** | **{clf_test_win['ep_accuracy']:.2%}** | **{clf_test_win['ep_bal_acc']:.2%}** | **{clf_test_win['ep_macro_f1']:.2%}** |",
    f"| Dummy Control (Test) | {clf_test_dum['win_accuracy']:.2%} | {clf_test_dum['win_bal_acc']:.2%} | {clf_test_dum['win_macro_f1']:.2%} | {clf_test_dum['ep_accuracy']:.2%} | {clf_test_dum['ep_bal_acc']:.2%} | {clf_test_dum['ep_macro_f1']:.2%} |",
    f"| Unsup Session-Norm Ref (Test) | {clf_test_unsup['win_accuracy']:.2%} | {clf_test_unsup['win_bal_acc']:.2%} | {clf_test_unsup['win_macro_f1']:.2%} | {clf_test_unsup['ep_accuracy']:.2%} | {clf_test_unsup['ep_bal_acc']:.2%} | {clf_test_unsup['ep_macro_f1']:.2%} |",
    "",
    "## Track C1: Biner Ekstrem Lintas-Peserta (`1–3` vs `7–10`)",
    f"- **Pemenang terkunci (Dev OOF):** `{bin_winner['candidate_id']}`",
    f"- **Dev OOF:** Window AUC = {bin_winner['win_auc']:.3f}, BalAcc = {bin_winner['win_bal_acc']:.2%} | Episode AUC = {bin_winner['ep_auc']:.3f}, BalAcc = {bin_winner['ep_bal_acc']:.2%}",
    f"- **Held-Out Test:** Window AUC = **{bin_test_met['win_auc']:.3f}**, BalAcc = **{bin_test_met['win_bal_acc']:.2%}** | Episode AUC = **{bin_test_met['ep_auc']:.3f}**, BalAcc = **{bin_test_met['ep_bal_acc']:.2%}**",
    "",
    "## Track C2: Diagnostik Kalibrasi Dalam-Peserta (Leave-One-Episode-Out)",
    "",
    "| Model | Peserta layak | Dummy ep-MAE | LOEO ep-MAE | Median ep-Spearman $\rho$ | Peserta $\rho > 0$ | Mean Binary AUC |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for m_name, s in wp_summary.items():
    auc_s = f"{s['mean_binary_auc']:.3f}" if s["mean_binary_auc"] is not None else "—"
    md.append(
        f"| {m_name} | {s['n_participants']} | {s['mean_ep_dummy_mae']:.3f} | {s['mean_ep_loeo_mae']:.3f} | {s['median_ep_spearman']:+.3f} | {s['positive_spearman_count']}/{s['n_participants']} | {auc_s} |"
    )
md += [
    "",
    "![V4 Overview](figures/v4_experiment_overview.png)",
    "",
    "![V4 Track B Ablation](figures/v4_ablation_trackB.png)",
]
(OUTPUT_DIR / "RESULTS.md").write_text("\n".join(md) + "\n", encoding="utf-8")
print("Artefak V4 lengkap tersimpan di:", OUTPUT_DIR)


Artefak V4 lengkap tersimpan di: E:\Skripsi\ngodong\Adaptive-IDE\models\batch-06\outputs\exp02_v4_self_reported_rating
